02_data_preparation

In [12]:
from pathlib import Path
import pandas as pd
from IPython.display import display

SEED = 42
DATA_PATH = Path("heart_2022_with_nans.csv")

raw_df = pd.read_csv(DATA_PATH)

print("Original dataset shape:", raw_df.shape)
display(raw_df.head())

Original dataset shape: (445132, 40)


,State,Sex,GeneralHealth,PhysicalHealthDays,MentalHealthDays,LastCheckupTime,PhysicalActivities,SleepHours,RemovedTeeth,HadHeartAttack,...,HeightInMeters,WeightInKilograms,BMI,AlcoholDrinkers,HIVTesting,FluVaxLast12,PneumoVaxEver,TetanusLast10Tdap,HighRiskLastYear,CovidPos
0,Alabama,Female,Very good,0.0,0.0,Within past year (anytime less than 12 months ...,No,8.0,NaN,No,...,NaN,NaN,NaN,No,No,Yes,No,"Yes, received tetanus shot but not sure what type",No,No
1,Alabama,Female,Excellent,0.0,0.0,NaN,No,6.0,NaN,No,...,1.60,68.04,26.57,No,No,No,No,"No, did not receive any tetanus shot in the pa...",No,No
2,Alabama,Female,Very good,2.0,3.0,Within past year (anytime less than 12 months ...,Yes,5.0,NaN,No,...,1.57,63.50,25.61,No,No,No,No,NaN,No,Yes
3,Alabama,Female,Excellent,0.0,0.0,Within past year (anytime less than 12 months ...,Yes,7.0,NaN,No,...,1.65,63.50,23.30,No,No,Yes,Yes,"No, did not receive any tetanus shot in the pa...",No,No
4,Alabama,Female,Fair,2.0,0.0,Within past year (anytime less than 12 months ...,Yes,9.0,NaN,No,...,1.57,53.98,21.77,Yes,No,No,Yes,"No, did not receive any tetanus shot in the pa...",No,No


In [13]:
# Keep the original data unchanged
data = raw_df.copy()

# Identify exact matches across all original columns
original_columns = raw_df.columns.tolist()

group_ids = raw_df.groupby(
    original_columns,
    dropna=False,
    sort=False,
    observed=True
).ngroup()

# Add identifiers for tracking only
data.insert(0, "row_id", range(len(data)))
data.insert(1, "duplicate_group", group_ids.to_numpy())

print("Total rows:", len(data))
print("Unique record groups:", data["duplicate_group"].nunique())
print("Extra identical rows:", raw_df.duplicated().sum())

Total rows: 445132
Unique record groups: 444975
Extra identical rows: 157


In [14]:
# Validate the target-source categories
target_sources = ["HadHeartAttack", "HadAngina"]

for column in target_sources:
    unexpected = set(data[column].dropna().unique()) - {"Yes", "No"}

    if unexpected:
        raise ValueError(
            f"Unexpected values in {column}: {unexpected}"
        )

# Positive if either response is Yes
positive = (
    data["HadHeartAttack"].eq("Yes")
    | data["HadAngina"].eq("Yes")
)

# Negative only if both responses are No
negative = (
    data["HadHeartAttack"].eq("No")
    & data["HadAngina"].eq("No")
)

data["heart_disease"] = pd.Series(
    pd.NA,
    index=data.index,
    dtype="Int64"
)

data.loc[positive, "heart_disease"] = 1
data.loc[negative, "heart_disease"] = 0

print("Positive targets:", int(positive.sum()))
print("Negative targets:", int(negative.sum()))
print("Unknown targets:", int(data["heart_disease"].isna().sum()))

Positive targets: 39751
Negative targets: 400360
Unknown targets: 5021


In [15]:
# Keep excluded row IDs for documentation
excluded_rows = data.loc[
    data["heart_disease"].isna(),
    ["row_id"]
].copy()

excluded_rows["reason"] = "Unknown heart disease target"

# Retain only records with a known target
prepared_data = data.loc[
    data["heart_disease"].notna()
].copy()

prepared_data["heart_disease"] = (
    prepared_data["heart_disease"].astype(int)
)

# Keep identifiers separate from model predictors
metadata = prepared_data[
    ["row_id", "duplicate_group"]
].copy()

# Exclude the target, target-source columns and identifiers
X = prepared_data.drop(
    columns=[
        "row_id",
        "duplicate_group",
        "heart_disease",
        "HadHeartAttack",
        "HadAngina"
    ]
)

y = prepared_data["heart_disease"].copy()

assert len(X) == len(y) == len(metadata)
assert not X.columns.isin(
    ["row_id", "duplicate_group", "heart_disease",
     "HadHeartAttack", "HadAngina"]
).any()

print("Rows retained:", len(prepared_data))
print("Rows excluded:", len(excluded_rows))
print("Number of candidate predictors:", X.shape[1])
print("Positive target percentage:", f"{y.mean():.2%}")

Rows retained: 440111
Rows excluded: 5021
Number of candidate predictors: 38
Positive target percentage: 9.03%


In [16]:
 # 70% training, 15% validation and 15% test

In [17]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [18]:
from sklearn.model_selection import train_test_split

# Confirm that every identical-record group has one target value
group_target_counts = prepared_data.groupby(
    "duplicate_group"
)["heart_disease"].nunique()

assert group_target_counts.eq(1).all()

# Create one entry per identical-record group
groups = prepared_data[
    ["duplicate_group", "heart_disease"]
].drop_duplicates("duplicate_group")

# Assign approximately 70% of groups to training
train_groups, remaining_groups = train_test_split(
    groups,
    test_size=0.30,
    random_state=SEED,
    stratify=groups["heart_disease"]
)

# Divide the remaining groups equally
validation_groups, test_groups = train_test_split(
    remaining_groups,
    test_size=0.50,
    random_state=SEED,
    stratify=remaining_groups["heart_disease"]
)

# Map group assignments back to individual rows
split_map = pd.concat([
    train_groups.assign(split="train"),
    validation_groups.assign(split="validation"),
    test_groups.assign(split="test")
]).set_index("duplicate_group")["split"]

split_manifest = metadata.copy()
split_manifest["split"] = split_manifest[
    "duplicate_group"
].map(split_map)

print("Group assignments created.")

Group assignments created.


In [19]:
# Check that every retained row has exactly one assignment
assert split_manifest["row_id"].is_unique
assert split_manifest["split"].notna().all()
assert len(split_manifest) == len(prepared_data)

# Check that identical records never appear in different splits
assert split_manifest.groupby(
    "duplicate_group"
)["split"].nunique().eq(1).all()

# Summarize row counts and target balance
split_audit = split_manifest.join(y)

split_summary = split_audit.groupby("split").agg(
    rows=("row_id", "size"),
    positive_cases=("heart_disease", "sum"),
    positive_pct=("heart_disease", "mean")
)

split_summary["negative_cases"] = (
    split_summary["rows"] - split_summary["positive_cases"]
)

split_summary["positive_pct"] *= 100
split_summary["dataset_pct"] = (
    split_summary["rows"] / len(prepared_data) * 100
)

split_summary = split_summary.reindex(
    ["train", "validation", "test"]
)

display(split_summary.round(2))
print("All split checks passed.")

,rows,positive_cases,positive_pct,negative_cases,dataset_pct
split,,,,,
train,308080,27826,9.03,280254,70.0
validation,66016,5963,9.03,60053,15.0
test,66015,5962,9.03,60053,15.0


All split checks passed.


In [20]:
OUTPUT_DIR = Path("data") / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save the shared row assignments
split_manifest.to_csv(
    OUTPUT_DIR / "split_manifest.csv",
    index=False
)

# Save excluded row IDs and their exclusion reason
excluded_rows.to_csv(
    OUTPUT_DIR / "excluded_rows.csv",
    index=False
)

# Save the split summary
split_summary.to_csv(
    OUTPUT_DIR / "split_summary.csv",
    index_label="split"
)

# Prepare raw-valued training data for the next stage
train_mask = split_manifest["split"].eq("train")

X_train = X.loc[train_mask].copy()
y_train = y.loc[train_mask].copy()

print("Files saved in:", OUTPUT_DIR.resolve())
print("Training predictors shape:", X_train.shape)
print("Training target shape:", y_train.shape)

Files saved in: C:\Users\QS\Desktop\UDST\DSAI5101\Project\New folder\Heart_Disease_Project\data\processed
Training predictors shape: (308080, 38)
Training target shape: (308080,)


In [21]:
 # training

In [22]:
# Audit missing values in training predictors only
training_missing = pd.DataFrame({
    "missing_count": X_train.isna().sum(),
    "missing_pct": X_train.isna().mean() * 100,
    "data_type": X_train.dtypes.astype(str)
}).sort_values("missing_count", ascending=False)

display(training_missing.round(2))

print(
    "Training rows with at least one missing value:",
    int(X_train.isna().any(axis=1).sum())
)

print(
    "Completely missing training columns:",
    X_train.columns[X_train.isna().all()].tolist()
)

# Save the audit
training_missing.to_csv(
    OUTPUT_DIR / "training_missing_summary.csv",
    index_label="feature"
)

,missing_count,missing_pct,data_type
TetanusLast10Tdap,56681,18.40,object
PneumoVaxEver,53001,17.20,object
HIVTesting,45404,14.74,object
ChestScan,38383,12.46,object
CovidPos,34810,11.30,object
HighRiskLastYear,34754,11.28,object
BMI,33328,10.82,float64
FluVaxLast12,32327,10.49,object
AlcoholDrinkers,31941,10.37,object
WeightInKilograms,28728,9.32,float64


Training rows with at least one missing value: 135348
Completely missing training columns: []


In [23]:
# Summarize numeric training predictors
display(X_train.describe().T.round(2))

# Flag values for review without modifying the data
range_flags = {
    "PhysicalHealthDays_outside_0_30": (
        X_train["PhysicalHealthDays"].notna()
        & ~X_train["PhysicalHealthDays"].between(0, 30)
    ),
    "MentalHealthDays_outside_0_30": (
        X_train["MentalHealthDays"].notna()
        & ~X_train["MentalHealthDays"].between(0, 30)
    ),
    "SleepHours_outside_0_24": (
        X_train["SleepHours"].notna()
        & ~X_train["SleepHours"].between(0, 24)
    ),
    "HeightInMeters_non_positive": X_train["HeightInMeters"].le(0),
    "WeightInKilograms_non_positive": X_train["WeightInKilograms"].le(0),
    "BMI_non_positive": X_train["BMI"].le(0)
}

range_summary = pd.Series(
    {name: int(mask.sum()) for name, mask in range_flags.items()},
    name="flagged_rows"
)

display(range_summary)

range_summary.to_csv(
    OUTPUT_DIR / "training_range_flags.csv",
    index_label="check"
)

,count,mean,std,min,25%,50%,75%,max
PhysicalHealthDays,300802.0,4.31,8.65,0.00,0.00,0.00,3.00,30.00
MentalHealthDays,302038.0,4.36,8.36,0.00,0.00,0.00,5.00,30.00
SleepHours,304555.0,7.03,1.50,1.00,6.00,7.00,8.00,24.00
HeightInMeters,288660.0,1.70,0.11,0.91,1.63,1.70,1.78,2.41
WeightInKilograms,279352.0,83.10,21.46,22.68,68.04,80.74,95.25,292.57
BMI,274752.0,28.53,6.55,12.06,24.13,27.44,31.75,99.34


PhysicalHealthDays_outside_0_30    0
MentalHealthDays_outside_0_30      0
SleepHours_outside_0_24            0
HeightInMeters_non_positive        0
WeightInKilograms_non_positive     0
BMI_non_positive                   0
Name: flagged_rows, dtype: int64

In [24]:
# Missing # keep BMI # remove height and weight

In [25]:
# Select predictors for the primary model
model_columns = X_train.columns.drop(
    ["HeightInMeters", "WeightInKilograms"]
).tolist()

X_train_model = X_train[model_columns].copy()

numeric_columns = X_train_model.select_dtypes(
    include="number"
).columns.tolist()

categorical_columns = [
    column for column in model_columns
    if column not in numeric_columns
]

print("Number of model predictors:", len(model_columns))
print("Numeric predictors:", numeric_columns)
print("Number of categorical predictors:", len(categorical_columns))

Number of model predictors: 36
Numeric predictors: ['PhysicalHealthDays', 'MentalHealthDays', 'SleepHours', 'BMI']
Number of categorical predictors: 32


In [26]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# Numeric values: training median plus missingness indicators
numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    )
])

# Categorical values: explicit Missing category plus one-hot encoding
categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value="Missing"
        )
    ),
    (
        "encoder",
        OneHotEncoder(handle_unknown="ignore")
    )
])

# Combine numeric and categorical preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_columns),
        ("categorical", categorical_pipeline, categorical_columns)
    ],
    sparse_threshold=1.0
)

# Convert categorical columns to object and standardize missing markers
X_train_model = X_train_model.astype({
    column: "object" for column in categorical_columns
})
X_train_model = X_train_model.where(
    X_train_model.notna(),
    float("nan")
)

# Learn preprocessing settings from training data only
X_train_processed = preprocessor.fit_transform(X_train_model)

print("Training shape before preprocessing:", X_train_model.shape)
print("Training shape after preprocessing:", X_train_processed.shape)
print("Preprocessing completed successfully.")

Training shape before preprocessing: (308080, 36)
Training shape after preprocessing: (308080, 184)
Preprocessing completed successfully.


In [27]:
# check all right and valid

In [28]:
import numpy as np
from scipy import sparse

# Check stored values without converting a sparse matrix to dense
values = (
    X_train_processed.data
    if sparse.issparse(X_train_processed)
    else np.asarray(X_train_processed)
)

assert np.isfinite(values).all(), (
    "Processed training data contains missing or infinite values."
)

assert X_train_processed.shape[0] == len(y_train), (
    "Predictors and target have different row counts."
)

# Save the names of the encoded features
feature_names = preprocessor.get_feature_names_out()

assert len(feature_names) == X_train_processed.shape[1]

pd.DataFrame({
    "feature_name": feature_names
}).to_csv(
    OUTPUT_DIR / "processed_feature_names.csv",
    index=False
)

print("All preprocessing checks passed.")
print("Number of processed features:", len(feature_names))

All preprocessing checks passed.
Number of processed features: 184


In [29]:
# src/prepare_data

In [30]:
from pathlib import Path

Path("docs").mkdir(exist_ok=True)
Path("src").mkdir(exist_ok=True)

print("Documentation and source folders are ready.")

Documentation and source folders are ready.


In [31]:
# Create a draft dictionary for all original columns
dictionary_rows = []

for column in raw_df.columns:
    series = raw_df[column]
    is_numeric = pd.api.types.is_numeric_dtype(series)

    if is_numeric:
        observed_values = (
            f"Observed range: {series.min()} to {series.max()}"
        )
    else:
        observed_values = " | ".join(
            sorted(series.dropna().astype(str).unique())
        )

    dictionary_rows.append({
        "column": column,
        "meaning": "TO COMPLETE: verify against source documentation",
        "variable_type": (
            "Numeric: verify meaning"
            if is_numeric
            else "Categorical: verify nominal or ordinal"
        ),
        "unit": "TO COMPLETE",
        "observed_values": observed_values,
        "allowed_values": "TO COMPLETE: verify against source",
        "raw_missing_count": int(series.isna().sum()),
        "missing_meaning": (
            "TO VERIFY: do not assume all missing responses "
            "have the same reason"
        ),
        "source_definition": "TO COMPLETE: document link and field",
        "role": (
            "Target source: excluded from predictors and Q2"
            if column in ["HadHeartAttack", "HadAngina"]
            else "Candidate predictor"
        )
    })

data_dictionary = pd.DataFrame(dictionary_rows)

data_dictionary.to_csv(
    "docs/data_dictionary.csv",
    index=False,
    encoding="utf-8-sig"
)

display(data_dictionary.head())
print("Draft saved: docs/data_dictionary.csv")

,column,meaning,variable_type,unit,observed_values,allowed_values,raw_missing_count,missing_meaning,source_definition,role
0,State,TO COMPLETE: verify against source documentation,Categorical: verify nominal or ordinal,TO COMPLETE,Alabama | Alaska | Arizona | Arkansas | Califo...,TO COMPLETE: verify against source,0,TO VERIFY: do not assume all missing responses...,TO COMPLETE: document link and field,Candidate predictor
1,Sex,TO COMPLETE: verify against source documentation,Categorical: verify nominal or ordinal,TO COMPLETE,Female | Male,TO COMPLETE: verify against source,0,TO VERIFY: do not assume all missing responses...,TO COMPLETE: document link and field,Candidate predictor
2,GeneralHealth,TO COMPLETE: verify against source documentation,Categorical: verify nominal or ordinal,TO COMPLETE,Excellent | Fair | Good | Poor | Very good,TO COMPLETE: verify against source,1198,TO VERIFY: do not assume all missing responses...,TO COMPLETE: document link and field,Candidate predictor
3,PhysicalHealthDays,TO COMPLETE: verify against source documentation,Numeric: verify meaning,TO COMPLETE,Observed range: 0.0 to 30.0,TO COMPLETE: verify against source,10927,TO VERIFY: do not assume all missing responses...,TO COMPLETE: document link and field,Candidate predictor
4,MentalHealthDays,TO COMPLETE: verify against source documentation,Numeric: verify meaning,TO COMPLETE,Observed range: 0.0 to 30.0,TO COMPLETE: verify against source,9067,TO VERIFY: do not assume all missing responses...,TO COMPLETE: document link and field,Candidate predictor


Draft saved: docs/data_dictionary.csv


In [32]:
decision_log = """# Data Preparation Decision Log

## Decisions implemented

1. Preserve the original CSV unchanged.
   Reason: retain a reproducible source dataset.

2. Create a stable row_id from the original CSV row order.
   Reason: track records and share split assignments.
   The row_id is not a model predictor.

3. Define heart_disease as 1 when HadHeartAttack or HadAngina
   is Yes, and 0 only when both are No.
   Reason: follow the project target definition.

4. Exclude 5,021 records with unknown targets.
   Reason: their outcome cannot be determined.
   Do not impute target labels.

5. Exclude HadHeartAttack and HadAngina from predictors and Q2.
   Reason: both fields define the target.

6. Retain identical records initially.
   Reason: identical survey answers do not prove duplicate people.
   Keep each exact-record group within one split.
   Group identifiers are not model predictors.

7. Split record groups using stratification and seed 42.
   Assign approximately 70% training, 15% validation and 15% test.
   Save row assignments in data/processed/split_manifest.csv.

8. Retain rows with missing predictor values.
   Reason: 135,348 training rows have at least one missing predictor.
   Complete-case deletion would discard about 43.9% of training rows.

9. Use training-median numeric imputation with missingness indicators.
   Use an explicit Missing category and one-hot encoding for categories.
   Learn preprocessing settings from training data only.

10. Keep BMI and exclude height and weight from the primary model.
    Reason: reduce overlap between BMI and its component measurements.
    Retain original measurements for EDA and sensitivity analysis.

11. Do not automatically remove extreme numeric values.
    The implemented range checks found no flagged training records.
    These checks do not establish that every extreme value is valid.

## Remaining work

- Verify all variable definitions, units and special response categories.
- Investigate training missingness by age, sex and state.
- Inspect fields that tend to be missing together.
- Compare records with known and unknown targets for exclusion context.
- Review extreme body measurements against source documentation.
- Coordinate numeric scaling for logistic regression with Member 3.
- Fit preprocessing within each cross-validation training fold.
- Compare alternative preprocessing on validation data with Member 3.
- Reserve test evaluation until model decisions are frozen.
"""

Path("docs/decision_log.md").write_text(
    decision_log,
    encoding="utf-8"
)

print("Saved: docs/decision_log.md")

Saved: docs/decision_log.md


In [33]:
%%writefile src/prepare_data.py
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split


def prepare_data(csv_path, seed=42):
    """Create known-target data and duplicate-aware split assignments."""
    raw = pd.read_csv(csv_path)

    for column in ["HadHeartAttack", "HadAngina"]:
        unexpected = set(raw[column].dropna().unique()) - {"Yes", "No"}
        if unexpected:
            raise ValueError(
                f"Unexpected values in {column}: {unexpected}"
            )

    group_ids = raw.groupby(
        raw.columns.tolist(),
        dropna=False,
        sort=False,
        observed=True
    ).ngroup()

    data = raw.copy()
    data.insert(0, "row_id", range(len(data)))
    data.insert(1, "duplicate_group", group_ids.to_numpy())

    positive = (
        data["HadHeartAttack"].eq("Yes")
        | data["HadAngina"].eq("Yes")
    )
    negative = (
        data["HadHeartAttack"].eq("No")
        & data["HadAngina"].eq("No")
    )

    data["heart_disease"] = pd.Series(
        pd.NA, index=data.index, dtype="Int64"
    )
    data.loc[positive, "heart_disease"] = 1
    data.loc[negative, "heart_disease"] = 0

    excluded = data.loc[
        data["heart_disease"].isna(), ["row_id"]
    ].copy()
    excluded["reason"] = "Unknown heart disease target"

    known = data.loc[data["heart_disease"].notna()].copy()
    known["heart_disease"] = known["heart_disease"].astype(int)

    assert known.groupby(
        "duplicate_group"
    )["heart_disease"].nunique().eq(1).all()

    groups = known[
        ["duplicate_group", "heart_disease"]
    ].drop_duplicates("duplicate_group")

    train_groups, remaining_groups = train_test_split(
        groups,
        test_size=0.30,
        random_state=seed,
        stratify=groups["heart_disease"]
    )

    validation_groups, test_groups = train_test_split(
        remaining_groups,
        test_size=0.50,
        random_state=seed,
        stratify=remaining_groups["heart_disease"]
    )

    assignments = pd.concat([
        train_groups.assign(split="train"),
        validation_groups.assign(split="validation"),
        test_groups.assign(split="test")
    ]).set_index("duplicate_group")["split"]

    manifest = known[["row_id", "duplicate_group"]].copy()
    manifest["split"] = manifest["duplicate_group"].map(assignments)

    assert manifest["row_id"].is_unique
    assert manifest["split"].notna().all()
    assert manifest.groupby(
        "duplicate_group"
    )["split"].nunique().eq(1).all()

    X = known.drop(columns=[
        "row_id", "duplicate_group", "heart_disease",
        "HadHeartAttack", "HadAngina"
    ])
    y = known["heart_disease"].copy()

    return X, y, manifest, excluded


def save_assignments(manifest, excluded, output_dir):
    """Save row assignments and exclusion reasons."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    manifest.to_csv(output_dir / "split_manifest.csv", index=False)
    excluded.to_csv(output_dir / "excluded_rows.csv", index=False)


if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser()
    parser.add_argument("csv_path")
    parser.add_argument("--output-dir", default="data/processed")
    args = parser.parse_args()

    X, y, manifest, excluded = prepare_data(args.csv_path)
    save_assignments(manifest, excluded, args.output_dir)

    print("Retained rows:", len(X))
    print("Excluded rows:", len(excluded))
    print("Preparation completed successfully.")

Writing src/prepare_data.py


In [34]:
import importlib.util

spec = importlib.util.spec_from_file_location(
    "project_preparation",
    Path("src/prepare_data.py")
)

preparation_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(preparation_module)

X_check, y_check, manifest_check, excluded_check = (
    preparation_module.prepare_data(DATA_PATH, seed=SEED)
)

pd.testing.assert_frame_equal(manifest_check, split_manifest)
pd.testing.assert_frame_equal(X_check, X)
pd.testing.assert_series_equal(y_check, y)
pd.testing.assert_frame_equal(excluded_check, excluded_rows)

print("The preparation script reproduces the notebook results.")

The preparation script reproduces the notebook results.
